In [ ]:
print("Hello, Jupyter!")

,Activity Date,Process Date,Settle Date,Instrument,Description,Trans Code,Quantity,Price,Amount
0,1/22/2026,1/22/2026,1/23/2026,QQQ,QQQ 1/23/2026 Put $617.00,STC,1,$0.73,$72.95
1,1/22/2026,1/22/2026,1/23/2026,QQQ,QQQ 1/23/2026 Put $617.00,STC,1,$0.75,$74.95
2,1/22/2026,1/22/2026,1/23/2026,SPY,SPY 1/22/2026 Put $689.00,STC,1,$0.43,$42.95
3,1/22/2026,1/22/2026,1/23/2026,SPY,SPY 1/22/2026 Put $689.00,BTO,1,$0.59,($59.04)
4,1/22/2026,1/22/2026,1/23/2026,QQQ,QQQ 1/23/2026 Put $617.00,BTO,1,$1.04,($104.04)
...,...,...,...,...,...,...,...,...,...
750,1/2/2026,1/2/2026,1/5/2026,SPY,SPY 1/5/2026 Put $686.00,BTO,5,$1.73,($865.10)
751,1/2/2026,1/2/2026,1/5/2026,ALAB,ALAB 1/16/2026 Call $200.00,STC,1,$3.35,$334.97
752,1/2/2026,1/2/2026,1/5/2026,NVO,NVO 1/15/2027 Call $55.00,STC,1,$8.15,$814.97
753,1/2/2026,1/2/2026,1/5/2026,ALAB,ALAB 1/16/2026 Call $200.00,STC,1,$2.85,$284.97


In [7]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import json
from datetime import datetime, timedelta
import os
import glob

# App Config
st.set_page_config(page_title="Ultimate Trading Journal Dashboard", layout="wide", page_icon="📈")
st.title("Ultimate Trading Journal Dashboard - @paddleurway")
st.markdown("""
This dashboard loads **all CSV files** from the current working folder automatically  
and combines them into one dataset.  
No need to upload — just place your exported .csv files in the same folder as this script.
""")

# ── Automatically find and load ALL CSV files in the current folder ──
@st.cache_data
def load_all_csvs():
    folder = os.getcwd()  # current working directory
    csv_files = glob.glob(os.path.join(folder, "*.csv"))
    
    if not csv_files:
        st.error("No .csv files found in the current folder.")
        st.info(f"Current folder: {folder}\nPut your exported CSV files here.")
        return pd.DataFrame()
    
    st.write(f"Found {len(csv_files)} CSV files:")
    for f in csv_files:
        st.write(f"• {os.path.basename(f)}")
    
    combined = []
    for file_path in csv_files:
        try:
            temp_df = pd.read_csv(
                file_path,
                on_bad_lines='warn',
                encoding='utf-8'
            )
            combined.append(temp_df)
        except Exception as e:
            st.warning(f"Could not load {os.path.basename(file_path)} → {e}")
    
    if not combined:
        st.error("No valid CSV files could be loaded.")
        return pd.DataFrame()
    
    df = pd.concat(combined, ignore_index=True)
    st.success(f"Combined {len(df)} rows from all CSVs.")
    
    return df

# Load data once
df = load_all_csvs()

if df.empty:
    st.stop()

# ── Data Cleaning ────────────────────────────────────────────────────────
df.columns = df.columns.str.strip()
df['Activity Date'] = pd.to_datetime(df['Activity Date'], errors='coerce')
df['Amount'] = pd.to_numeric(
    df['Amount'].astype(str).str.replace(r'[\$,()]', '', regex=True),
    errors='coerce'
)
df = df.sort_values('Activity Date').reset_index(drop=True)

# Derived columns
df['Type'] = df['Description'].str.contains('Put', case=False, na=False).map({True: 'Put', False: 'Call'})
df['Week Day'] = df['Activity Date'].dt.day_name()
df['Month'] = df['Activity Date'].dt.month_name()
df['Hour'] = df['Activity Date'].dt.hour
df['Cum PL'] = df['Amount'].cumsum()
df['Duration'] = df['Activity Date'].diff().dt.total_seconds() / 3600
df['Duration'] = df['Duration'].fillna(0)

# ── Sidebar: Filters ─────────────────────────────────────────────────────
st.sidebar.header("Filters & Settings")

instruments = sorted(df['Instrument'].dropna().unique())
selected_instr = st.sidebar.multiselect("Instruments", instruments, default=instruments)

date_min = df['Activity Date'].min().date() if pd.notna(df['Activity Date'].min()) else datetime.today().date()
date_max = df['Activity Date'].max().date() if pd.notna(df['Activity Date'].max()) else datetime.today().date()

start_date = st.sidebar.date_input("Start Date", value=date_min, min_value=date_min, max_value=date_max)
end_date   = st.sidebar.date_input("End Date",   value=date_max, min_value=date_min, max_value=date_max)

# Filter
filtered_df = df[
    (df['Instrument'].isin(selected_instr)) &
    (df['Activity Date'].dt.date >= start_date) &
    (df['Activity Date'].dt.date <= end_date)
].copy()

# ── Metrics Function ─────────────────────────────────────────────────────
def calculate_metrics(df):
    if df.empty:
        return {}
    total_pl = df['Amount'].sum()
    trades = len(df)
    wins = df[df['Amount'] > 0]
    losses = df[df['Amount'] < 0]
    win_count = len(wins)
    loss_count = len(losses)
    win_rate = (win_count / trades * 100) if trades > 0 else 0
    avg_win = wins['Amount'].mean() if win_count > 0 else 0
    avg_loss = losses['Amount'].mean() if loss_count > 0 else 0
    profit_factor = abs(wins['Amount'].sum() / losses['Amount'].sum()) if loss_count > 0 else np.inf
    max_drawdown = (df['Cum PL'] - df['Cum PL'].cummax()).min()
    expectancy = (win_rate / 100 * avg_win) + ((1 - win_rate / 100) * avg_loss)
    r_multiple = (avg_win / abs(avg_loss)) if avg_loss != 0 else np.inf
    sharpe = (df['Amount'].mean() / df['Amount'].std()) * np.sqrt(252) if df['Amount'].std() != 0 else 0
    sortino = (df['Amount'].mean() / losses['Amount'].std()) * np.sqrt(252) if losses['Amount'].std() != 0 else 0
    longest_win = max((df['Amount'] > 0).cumsum().groupby((df['Amount'] <= 0).cumsum()).max()) if win_count > 0 else 0
    longest_loss = max((df['Amount'] <= 0).cumsum().groupby((df['Amount'] > 0).cumsum()).max()) if loss_count > 0 else 0
    avg_duration = df['Duration'].mean()
    total_volume = df['Quantity'].sum() if 'Quantity' in df else 0
    recovery_factor = total_pl / abs(max_drawdown) if max_drawdown != 0 else np.inf

    return {
        'Total P/L': total_pl,
        'Win Rate %': win_rate,
        'Avg Win': avg_win,
        'Avg Loss': avg_loss,
        'Profit Factor': profit_factor,
        'Max Drawdown': max_drawdown,
        'Expectancy': expectancy,
        'R-Multiple': r_multiple,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Longest Win Streak': longest_win,
        'Longest Loss Streak': longest_loss,
        'Avg Holding Time (hrs)': avg_duration,
        'Total Volume': total_volume,
        'Recovery Factor': recovery_factor
    }

# ── Main Dashboard ───────────────────────────────────────────────────────
tab1, tab2, tab3, tab4 = st.tabs(["Overview", "Metrics", "Charts", "Raw Data"])

with tab1:
    st.header("Overview")
    if filtered_df.empty:
        st.warning("No data matches the selected filters.")
    else:
        metrics = calculate_metrics(filtered_df)
        cols = st.columns(4)
        cols[0].metric("Total P/L", f"${metrics['Total P/L']:,.2f}")
        cols[1].metric("Trades", len(filtered_df))
        cols[2].metric("Win Rate", f"{metrics['Win Rate %']:.1f}%")
        cols[3].metric("Expectancy", f"${metrics['Expectancy']:.2f}")
        
        fig_eq = px.line(filtered_df, x='Activity Date', y='Cum PL', title="Equity Curve")
        st.plotly_chart(fig_eq, use_container_width=True)

with tab2:
    st.header("Detailed Metrics")
    if filtered_df.empty:
        st.warning("No data.")
    else:
        metrics = calculate_metrics(filtered_df)
        metrics_df = pd.DataFrame(list(metrics.items()), columns=["Metric", "Value"])
        st.table(metrics_df.style.format({
            "Value": lambda x: f"${x:,.2f}" if isinstance(x, (int, float)) and "Rate" not in metrics_df["Metric"][metrics_df["Value"] == x].iloc[0] else "{:.2f}"
        }))

with tab3:
    st.header("Visualizations")
    if filtered_df.empty:
        st.warning("No data.")
    else:
        # P/L by Instrument
        pl_inst = filtered_df.groupby('Instrument')['Amount'].sum().reset_index()
        fig_bar = px.bar(pl_inst, x='Instrument', y='Amount', title="P/L by Instrument")
        st.plotly_chart(fig_bar, use_container_width=True)
        
        # Put vs Call
        type_cnt = filtered_df['Type'].value_counts().reset_index()
        fig_pie = px.pie(type_cnt, values='count', names='Type', title="Put vs Call")
        st.plotly_chart(fig_pie, use_container_width=True)
        
        # P/L Histogram
        fig_hist = px.histogram(filtered_df, x='Amount', title="P/L Distribution", color='Type')
        st.plotly_chart(fig_hist, use_container_width=True)

with tab4:
    st.header("Raw Trades Table")
    if filtered_df.empty:
        st.warning("No data.")
    else:
        st.dataframe(
            filtered_df.style.format({
                "Amount": "${:,.2f}",
                "Cum PL": "${:,.2f}",
                "Activity Date": "{:%Y-%m-%d %H:%M}"
            })
        )

# Footer
st.markdown("---")
st.caption("Automatically loads all .csv files from the folder where this script is running. Place your Numbers-exported CSVs here.")

2026-01-22 20:22:31.952 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-22 20:22:31.953 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-22 20:22:31.954 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-22 20:22:31.954 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-22 20:22:31.955 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-22 20:22:31.955 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-22 20:22:31.956 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-22 20:22:31.957 No runtime found, using MemoryCacheStorageManager
2026-01-22 20:22:31.959 No runtime found, us

IndexError: single positional indexer is out-of-bounds

In [8]:
df

,Activity Date,Process Date,Settle Date,Instrument,Description,Trans Code,Quantity,Price,Amount,Type,Week Day,Month,Hour,Cum PL,Duration
0,2026-01-02,1/2/2026,1/5/2026,ALAB,ALAB 1/16/2026 Call $200.00,STC,1,$2.85,284.97,Call,Friday,January,0.0,284.97,0.0
1,2026-01-02,1/2/2026,1/5/2026,SPY,SPY 1/2/2026 Put $681.00,STC,2,$0.72,143.95,Put,Friday,January,0.0,428.92,0.0
2,2026-01-02,1/2/2026,1/5/2026,SPY,SPY 1/2/2026 Put $681.00,BTO,3,$0.44,132.06,Put,Friday,January,0.0,560.98,0.0
3,2026-01-02,1/2/2026,1/5/2026,SPY,SPY 1/2/2026 Put $681.00,STC,1,$0.41,40.97,Put,Friday,January,0.0,601.95,0.0
4,2026-01-02,1/2/2026,1/5/2026,SPY,SPY 1/2/2026 Put $681.00,STC,1,$0.41,40.97,Put,Friday,January,0.0,642.92,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
750,2026-01-22,1/22/2026,1/23/2026,QQQ,QQQ 1/22/2026 Call $621.00,STC,1,$0.89,88.95,Call,Thursday,January,0.0,156184.99,0.0
751,2026-01-22,1/22/2026,1/23/2026,PYPL,PYPL 6/18/2026 Call $60.00,STC,2,$4.75,949.91,Call,Thursday,January,0.0,157134.90,0.0
752,2026-01-22,1/22/2026,1/23/2026,QQQ,QQQ 1/22/2026 Put $618.00,STC,2,$0.90,179.91,Put,Thursday,January,0.0,157314.81,0.0
753,2026-01-22,1/22/2026,1/23/2026,QQQ,QQQ 1/23/2026 Put $617.00,STC,1,$0.73,72.95,Put,Thursday,January,0.0,157387.76,0.0
